
# Операції з тензорами

Нейронні мережі будуються на матричних операціях, насамперед на матричному
добутку. Щоб зробити трансформер з нуля, нам постійно доведеться працювати з
векторами, матрицями та тензорами.

Матричний добуток — це фундаментальна операція. Саме на неї припадатиме більша
частина часу та обчислювальних ресурсів при використанні великих мовних
моделей.

Оскільки ця операція така важлива, є багато способів, як пришвидшити тензорний
добуток. Про це ми поговоримо в окремому курсі (LLM Systems).

Крім того, існує багато бібліотек для роботи з тензорами. У світі deep
learning та великих мовних моделей найпопулярнішою є PyTorch, яка в свою чергу
сильно надихалася API NumPy.

У реальному житті ви напевно будете використовувати саме PyTorch, але зараз ми
імплементуємо всі операції з нуля без використання сторонніх бібліотек.

Трохи термінології:
- тензор - багатовимірний числовий масив
- розмірність, ранг, порядок тензора - кількість осей
- скаляр - одне число, яке можна вважати 0-вимірним тензором
- вектор - 1-вимірний тензор
- матриця - 2-вимірний тензор

In [ ]:

def vector_add(a, b):
    """Додає два вектори поелементно."""
    
    ...

# Питання:
# - Скільки числових додавань використовується для векторів довжини n?
# - Скільки додаткової пам'яті потрібно для результату?


def test_vector_add():
    assert vector_add([1, 2, 3], [4, 5, 6]) == [5, 7, 9]
    assert vector_add([0.5, 1.5], [0.25, -0.5]) == [0.75, 1.0]
    assert vector_add([], []) == []

    try:
        vector_add([1, 2], [3, 4, 5])
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Додавання векторів різних довжин має давати помилку")


if __name__ == "__main__":
    test_vector_add()
    print("✓ vector_add")

In [ ]:

def dot(a, b):
    """Скалярний добуток двох векторів."""
    ...

# Питання:
# - Скільки операцій множення та додавання виконується для векторів довжиною n?
# - Якщо подвоїти довжину векторів, як зміниться кількість операцій?

def test_dot():
    assert dot([1, 2, 3], [4, 5, 6]) == 32
    assert dot([2, -3], [4, 0.5]) == 6.5
    assert dot([], []) == 0

    # Ненульові вектори теж можуть мати нульовий скалярний добуток.
    assert dot([1, 0], [0, 1]) == 0

    try:
        dot([1, 2], [3])
    except ValueError:
        pass
    else:
        raise AssertionError("Добуток векторів різних неможливий")


if __name__ == "__main__":
    test_dot()
    print("✓ dot")


## Множення матриці на вектор

Множення матриці не вектор це як `dot` для багатьох векторів одразу

Скалярний добуток двох векторів `dot(a, b)` дає одне число. Але часто буває
так, що у нас є один вхідний вектор `x`, і його треба перемножити не з одним,
а з багатьма різними векторами $a_1, a_2, \dots, a_m$.

Один варіант це написати цикл і викликати `dot` $m$ разів. А можна зробити
інакше. Складемо всі ці вектори рядками в одну матрицю

$$ A = \begin{bmatrix} a_1 \\ a_2 \\ \vdots \\ a_m \end{bmatrix} $$

і виконаємо одну операцію, множення матриці на вектор `matvec(A, x)`.
Результатом буде вектор з $m$ чисел. Перше з них дорівнює `dot(a_1, x)`, друге
дорівнює `dot(a_2, x)`, і так далі.

Тобто множення матриці на вектор є просто `dot`, застосованим до кожного
рядка.

### Приклад: кошик покупок

Ви купуєте 2 буханки хліба, 1 кг яблук і 3 л молока. Запишемо це як вектор.

```python
vec = [2, 1, 3]  # хліб, яблука, молоко
```

Ціни на ці товари є у двох магазинах. Кожен рядок матриці є прайсом одного
магазину, записаним у тому ж порядку (хліб, яблука, молоко).

```python
mat = [
    [1, 2, 3],  # магазин A: хліб=1, яблука=2, молоко=3
    [4, 5, 6],  # магазин B: хліб=4, яблука=5, молоко=6
]
```

Скільки ви заплатите в кожному магазині? Для одного магазину відповідь є `dot`
вашого кошика з його прайсом:

$$
\begin{aligned}
\text{магазин A:}\quad & 1 \cdot 2 + 2 \cdot 1 + 3 \cdot 3 = 13 \\
\text{магазин B:}\quad & 4 \cdot 2 + 5 \cdot 1 + 6 \cdot 3 = 31
\end{aligned}
$$

А для всіх магазинів одразу відповідь дає множення матриці на вектор:

```python
matvec(mat, vec) == [13, 31]
```

Той самий результат у вигляді вектора.

### Як про це думати

Вектор є вхідними даними (наш кошик). Кожен рядок матриці є окремим
«рецептом», як стиснути цей вхід в одне число (прайс конкретного магазину).
Результат є вектором, у якому по одному числу на кожен рядок (сума чеку в
кожному магазині).

Звідси одразу випливають вимоги до розмірів.

Рядок матриці має бути такої ж довжини, як вектор, інакше не буде з чим
перемножувати. Не можна помножити ціну молока на кількість товару, якого немає
в кошику.

Кількість рядків матриці задає довжину результату. Скільки «рецептів», стільки
й чисел на виході.

mat форми (m, n) * vec довжини n -> результат довжини m

У нашому прикладі (2, 3) * 3 -> 2. Три товари, два магазини, дві суми.

### Формально

$$ y_i = \sum_{j=0}^{n-1} A_{ij}\, x_j = \operatorname{dot}(A_{i,:},\ x) $$

де $A_{i,:}$ позначає весь рядок з індексом $i$. Це і є запис словами «для
кожного рядка $i$ порахувати `dot` цього рядка з вектором $x$».

In [ ]:

def matvec(mat, vec):
    """Матрично-векторний добуток."""
    
    # Використати dot
    ...
    

def test_matvec():
    assert matvec(
        [[1, 2, 3],
         [4, 5, 6]],
        [10, 20, 30],
    ) == [140, 320]
    
    # Одинична матриця не змінює вектор.
    assert matvec(
        [[1, 0],
         [0, 1]],
        [7, -3],
    ) == [7, -3]
    
    # Один рядок: результат усе одно вектор, а не скаляр.
    assert matvec([[2, 3]], [4, 5]) == [23]
    
    # Один стовпець.
    assert matvec([[2], [3], [4]], [10]) == [20, 30, 40]
    
    assert matvec([[0, 0], [0, 0]], [5, 6]) == [0, 0]
    assert matvec([[], []], []) == [0.0, 0.0]
    
    a = [[1, 2], [3, 4]]
    x = [5, 6]
    matvec(a, x)
    assert a == [[1, 2], [3, 4]]
    assert x == [5, 6]
    
    # Рядки матриці мають мати ту саму довжину, що й вектор.
    try:
        matvec([
            [1, 2],
            [3, 4]],
        [1])
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Очікується ValueError: довжина вектора не відповідає довжині рядків"
        )

    # Усі рядки матриці мають бути однакової довжини.
    try:
        matvec([
            [1, 2],
            [3]],
        [1, 2])
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Очікується ValueError: матриця має рядки різної довжини"
        )

    # Для порожньої матриці неможливо визначити кількість її стовпців.
    try:
        matvec([], [])
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Очікується ValueError: порожня матриця не визначає кількість стовпців"
        )
    
    print("Усі перевірки matvec пройдено.")


if __name__ == "__main__":
    test_matvec()
    print("✓ matvec")